In [0]:
%reload_ext autoreload
%autoreload 2

import sys
import os
from pyspark.sql.functions import col, isnan, when, count

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.data_prep._02_silver import clean_bronze_to_silver

CATALOG = "xaids_catalogue"
table_bronze = "cicds2017_raw_full"
table_silver = "cicds2017_cleaned"

df_silver = clean_bronze_to_silver(spark, CATALOG, table_bronze, table_silver)


Traitements Silver terminés. Table sauvegardée : xaids_catalogue.02_silver.cicds2017_cleaned


In [0]:
print("\n--- Comptage des valeurs Nulles par colonne ---")

colonnes_exclues = ["Label", "ingestion_timestamp", "source_file"]

exprs = []
for c in df_silver.columns:
    if c not in colonnes_exclues:
        exprs.append(count(when(col(c).isNull() | isnan(c), c)).alias(c))
        
null_counts = df_silver.select(exprs)
display(null_counts)


--- Comptage des valeurs Nulles par colonne ---


Destination_Port,Flow_Duration,Total_Fwd_Packets,Total_Backward_Packets,Total_Length_of_Fwd_Packets,Total_Length_of_Bwd_Packets,Fwd_Packet_Length_Max,Fwd_Packet_Length_Min,Fwd_Packet_Length_Mean,Fwd_Packet_Length_Std,Bwd_Packet_Length_Max,Bwd_Packet_Length_Min,Bwd_Packet_Length_Mean,Bwd_Packet_Length_Std,Flow_Bytes/s,Flow_Packets/s,Flow_IAT_Mean,Flow_IAT_Std,Flow_IAT_Max,Flow_IAT_Min,Fwd_IAT_Total,Fwd_IAT_Mean,Fwd_IAT_Std,Fwd_IAT_Max,Fwd_IAT_Min,Bwd_IAT_Total,Bwd_IAT_Mean,Bwd_IAT_Std,Bwd_IAT_Max,Bwd_IAT_Min,Fwd_PSH_Flags,Bwd_PSH_Flags,Fwd_URG_Flags,Bwd_URG_Flags,Fwd_Header_Length,Bwd_Header_Length,Fwd_Packets/s,Bwd_Packets/s,Min_Packet_Length,Max_Packet_Length,Packet_Length_Mean,Packet_Length_Std,Packet_Length_Variance,FIN_Flag_Count,SYN_Flag_Count,RST_Flag_Count,PSH_Flag_Count,ACK_Flag_Count,URG_Flag_Count,CWE_Flag_Count,ECE_Flag_Count,Down/Up_Ratio,Average_Packet_Size,Avg_Fwd_Segment_Size,Avg_Bwd_Segment_Size,Fwd_Avg_Bytes/Bulk,Fwd_Avg_Packets/Bulk,Fwd_Avg_Bulk_Rate,Bwd_Avg_Bytes/Bulk,Bwd_Avg_Packets/Bulk,Bwd_Avg_Bulk_Rate,Subflow_Fwd_Packets,Subflow_Fwd_Bytes,Subflow_Bwd_Packets,Subflow_Bwd_Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
display(df_silver.select("Label").distinct())

Label
BENIGN
Bot
FTP-Patator
Web Attack - XSS
Web Attack - Brute Force
SSH-Patator
Web Attack - Sql Injection
Infiltration
PortScan
DDoS


In [0]:
total_lignes = df_silver.count()
lignes_uniques = df_silver.dropDuplicates().count()
nb_doublons = total_lignes - lignes_uniques

print(f"Nombre total de lignes : {total_lignes}")
print(f"Nombre de lignes uniques : {lignes_uniques}")
print(f"Nombre de doublons : {nb_doublons}")

Nombre total de lignes : 2571305
Nombre de lignes uniques : 2571305
Nombre de doublons : 0


In [0]:
print("\n--- Distribution des attaques (Labels) ---")
display(df_silver.groupBy("Label").count().orderBy("count", ascending=False))


--- Distribution des attaques (Labels) ---


Label,count
BENIGN,2145621
DoS Hulk,172691
DDoS,127997
PortScan,90819
DoS GoldenEye,10281
FTP-Patator,5929
DoS slowloris,5385
DoS Slowhttptest,5228
SSH-Patator,3217
Bot,1953


In [0]:
display(df_silver.describe())

summary,Destination_Port,Flow_Duration,Total_Fwd_Packets,Total_Backward_Packets,Total_Length_of_Fwd_Packets,Total_Length_of_Bwd_Packets,Fwd_Packet_Length_Max,Fwd_Packet_Length_Min,Fwd_Packet_Length_Mean,Fwd_Packet_Length_Std,Bwd_Packet_Length_Max,Bwd_Packet_Length_Min,Bwd_Packet_Length_Mean,Bwd_Packet_Length_Std,Flow_Bytes/s,Flow_Packets/s,Flow_IAT_Mean,Flow_IAT_Std,Flow_IAT_Max,Flow_IAT_Min,Fwd_IAT_Total,Fwd_IAT_Mean,Fwd_IAT_Std,Fwd_IAT_Max,Fwd_IAT_Min,Bwd_IAT_Total,Bwd_IAT_Mean,Bwd_IAT_Std,Bwd_IAT_Max,Bwd_IAT_Min,Fwd_PSH_Flags,Bwd_PSH_Flags,Fwd_URG_Flags,Bwd_URG_Flags,Fwd_Header_Length,Bwd_Header_Length,Fwd_Packets/s,Bwd_Packets/s,Min_Packet_Length,Max_Packet_Length,Packet_Length_Mean,Packet_Length_Std,Packet_Length_Variance,FIN_Flag_Count,SYN_Flag_Count,RST_Flag_Count,PSH_Flag_Count,ACK_Flag_Count,URG_Flag_Count,CWE_Flag_Count,ECE_Flag_Count,Down/Up_Ratio,Average_Packet_Size,Avg_Fwd_Segment_Size,Avg_Bwd_Segment_Size,Fwd_Avg_Bytes/Bulk,Fwd_Avg_Packets/Bulk,Fwd_Avg_Bulk_Rate,Bwd_Avg_Bytes/Bulk,Bwd_Avg_Packets/Bulk,Bwd_Avg_Bulk_Rate,Subflow_Fwd_Packets,Subflow_Fwd_Bytes,Subflow_Bwd_Packets,Subflow_Bwd_Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label,source_file
count,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305,2571305
mean,8543.095024899807,1.6200363717496369E7,6.007396244319518,5.887405033630783,570.0299283826695,5548.158428891166,226.5372135938755,19.5047254215272,62.84929471191875,75.62868330936803,953.3074411631447,43.563143617734966,333.7945903108306,368.2695404511434,1439365.3188658312,48695.36868672888,1416805.7572957594,3210041.2100581867,1.0073553513545457E7,167160.07808797478,1.5868718767038528E7,2859791.7894938355,3591299.7604988124,9920872.415391017,1113350.816465569,1.0833612962350635E7,1986555.750555407,1632603.7889356518,5139353.158231326,1064755.7421099404,0.04812070135592627,0.0,3.111260624468898E-5,0.0,158.41748528470953,156.61314196487777,40840.996740278984,6591.589040836487,17.1730564829921,1040.2541219341929,187.0270790689051,323.09060252689915,533897.8759292098,0.03158551785960825,0.04812070135592627,2.6445715307985636E-4,0.29095070401994316,0.31039336056982736,0.10062789128477563,3.111260624468898E-5,2.656238758140322E-4,0.702999060788199,208.6288668880201,62.84929471191875,333.79459031084065,0.0,0.0,0.0,0.0,0.0,0.0,6.007396244319518,570.0299283826695,5.887405033630783,5548.146656269871,7130.689008888483,2185.7988585562584,3.022833541722977,25.888850992006006,88473.75721470377,44703.7259460771,166393.42130863512,63198.541114336884,9132473.854132984,553544.218648368,9549640.188959302,8697558.414136402,null,null
stddev,18881.77381732923,3.489773032786092E7,24.82721929227486,34.22670839288522,4824.249668679543,61741.230456922065,748.1858714310617,60.406930048258225,193.51227237343747,293.8694756167234,2019.3184466744897,70.69217236460338,626.3794886593231,873.6639113839778,2.6395427916622825E7,207410.97902872614,4641075.370444446,8383715.469178254,2.545469199013137E7,2984644.4668063303,3.482939744969899E7,9925277.491124256,1.005085542150256E7,2.553402402888167E7,8971464.759969417,2.9894033262331624E7,9304653.743671969,6565684.76292086,1.7917601116374925E7,8711688.331242852,0.21402130097468627,0.0,0.005577781848546807,0.0,639.2908018116333,823.8149192776957,192561.257699404,38146.6421519169,25.65482167070587,210

In [0]:
from pyspark.sql.functions import col, sum, when

df_silver = spark.read.table("xaids_catalogue.02_silver.cicds2017_cleaned")

# 1. Définition des listes de colonnes par catégorie
time_cols = ["Flow_Duration", "Flow_IAT_Mean", "Flow_IAT_Max", "Flow_IAT_Min", "Fwd_IAT_Min"]
rate_cols = ["Flow_Bytes_s", "Flow_Packets_s"]
underflow_cols = ["Fwd_Header_Length34", "Fwd_Header_Length55", "Bwd_Header_Length", "min_seg_size_forward"]
window_cols = ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]

# 2. Création des expressions d'agrégation (Comptage des lignes vérifiant la condition)
# On utilise sum(when(condition, 1).otherwise(0)) pour compter
audit_exprs = []

# Comptage total pour référence
import pyspark.sql.functions as F
audit_exprs.append(F.count("*").alias("Total_Lignes"))

# A. Temps négatifs (< 0)
for c in time_cols:
    if c in df_silver.columns:
        audit_exprs.append(sum(when(col(c) < 0, 1).otherwise(0)).alias(f"Err_Time_{c}"))

# B. Débits négatifs (< 0)
for c in rate_cols:
    if c in df_silver.columns:
        audit_exprs.append(sum(when(col(c) < 0, 1).otherwise(0)).alias(f"Err_Rate_{c}"))

# C. Underflows de mémoire (< 0)
for c in underflow_cols:
    if c in df_silver.columns:
        audit_exprs.append(sum(when(col(c) < 0, 1).otherwise(0)).alias(f"Err_Underflow_{c}"))

# D. Placeholders TCP (== -1)
for c in window_cols:
    if c in df_silver.columns:
        audit_exprs.append(sum(when(col(c) == -1, 1).otherwise(0)).alias(f"Info_Placeholder_{c}"))

# 3. Exécution de l'audit sur le DataFrame
df_audit = df_silver.select(*audit_exprs)

# Affichage des résultats
print("Rapport des valeurs aberrantes :")
display(df_audit)

Rapport des valeurs aberrantes :


Total_Lignes,Err_Time_Flow_Duration,Err_Time_Flow_IAT_Mean,Err_Time_Flow_IAT_Max,Err_Time_Flow_IAT_Min,Err_Time_Fwd_IAT_Min,Err_Underflow_Bwd_Header_Length,Err_Underflow_min_seg_size_forward,Info_Placeholder_Init_Win_bytes_forward,Info_Placeholder_Init_Win_bytes_backward
2571305,0,0,0,0,0,0,0,951457,1264818
